In [1]:
import os
import pandas as pd
from datasets import load_dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoConfig,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
)
import numpy as np
from sklearn.metrics import precision_recall_fscore_support, accuracy_score
from sklearn.model_selection import StratifiedKFold

c:\Users\c24082331\OneDrive - Cardiff University\Desktop\RA(UniversalCEFR)\development\universalcefr\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
ds = load_dataset("UniversalCEFR/learn_welsh_cy")["train"]

In [10]:
pd.DataFrame(ds)

,title,lang,source_name,format,category,cefr_level,license,text
0,Uned 1 - Sgwrs 1,cy,mynediad-de-learnwelsh,dialogue-level,reference,A1,public,"A: Helô, Eryl dw i. Pwy dych chi?\nB: Bore da,..."
1,Uned 1 - Sgwrs 2,cy,mynediad-de-learnwelsh,dialogue-level,reference,A1,public,"A: O na, yr heddlu! (Stopio'r car)\nB: Hello, ..."
2,Uned 1 - Sgwrs 3,cy,mynediad-de-learnwelsh,dialogue-level,reference,A1,public,"A: Bore da. Sut dych chi?\nB: Iawn, ond wedi b..."
3,Uned 2 - Sgwrs 1,cy,mynediad-de-learnwelsh,dialogue-level,reference,A1,public,"Ceri: Noswaith dda, Eryl. Sut wyt ti?\nEryl: D..."
4,Uned 2 - Sgwrs 2,cy,mynediad-de-learnwelsh,dialogue-level,reference,A1,public,A: Bore da.\nB: Hmff.\nA: Sut dych chi heddiw?...
...,...,...,...,...,...,...,...,...
1367,Uned 23 - na,cy,sylfaen-de-learnwelsh,sentence-level,reference,A2,public,Allech chi gyrraedd yn gynnar?
1368,Uned 23 - na,cy,sylfaen-de-learnwelsh,sentence-level,reference,A2,public,Allet ti gyrraedd yn gynnar?
1369,Uned 23 - na,cy,sylfaen-de-learnwelsh,sentence-level,reference,A2,public,Allai hi gyrraedd yn gynnar?
1370,Uned 23 - na,cy,sylfaen-de-learnwelsh,sentence-level,reference,A2,public,Allen nhw gyrraedd yn gynnar?


In [9]:
label2id = {lvl: i for i,lvl in enumerate(["A1","A2","B1","B2","C1","C2"])}
labels = np.array([label2id[l] for l in ds["cefr_level"]])

In [10]:
tokenizer = AutoTokenizer.from_pretrained("UniversalCEFR/xlm-roberta-base-cefr-all-classifier")
model = AutoModelForSequenceClassification.from_pretrained("UniversalCEFR/xlm-roberta-base-cefr-all-classifier")
data_collator = DataCollatorWithPadding(tokenizer)

In [11]:
def preprocess(batch):
    toks = tokenizer(batch["text"], truncation=True, max_length=256)
    toks["labels"] = [label2id[l] for l in batch["cefr_level"]]
    return toks

In [15]:
def compute_metrics(pred):
    logits, y = pred
    preds = np.argmax(logits, axis=-1)
    p,r,f1,_ = precision_recall_fscore_support(y, preds, average="weighted")
    return {"accuracy": accuracy_score (y, preds),"precision": p,"recall":r,"f1_weighted": f1}

In [17]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
all_results = []

In [18]:
for fold, (train_idx, val_idx) in enumerate(skf.split(ds, labels), start=1):
    print(f"\n--- Fold {fold} ---")
    # Slice out train/val
    ds_train = ds.select(train_idx)
    ds_val   = ds.select(val_idx)

    # Tokenize each
    tok_train = ds_train.map(preprocess, batched=True, remove_columns=ds_train.column_names)
    tok_val   = ds_val.map(preprocess,   batched=True, remove_columns=ds_val.column_names)


--- Fold 1 ---


Map: 100%|██████████| 275/275 [00:00<00:00, 10312.51 examples/s]



--- Fold 2 ---


Map: 100%|██████████| 275/275 [00:00<00:00, 9798.61 examples/s]



--- Fold 3 ---


Map: 100%|██████████| 274/274 [00:00<00:00, 9526.42 examples/s]



--- Fold 4 ---


Map: 100%|██████████| 274/274 [00:00<00:00, 10922.87 examples/s]



--- Fold 5 ---


Map: 100%|██████████| 274/274 [00:00<00:00, 10613.98 examples/s]


In [20]:
args = TrainingArguments(
    output_dir=f"./xlmr_cv_fold_{fold}",
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    eval_strategy="epoch",
    save_strategy="no",
    logging_steps=50,
    seed=42,
)

In [21]:
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tok_train,
    eval_dataset=tok_val,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

C:\Users\c24082331\AppData\Local\Temp\ipykernel_26712\3311683596.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [22]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1 Weighted
1,0.377400,1.165742,0.664234,0.753812,0.664234,0.650740
2,0.159100,0.672585,0.868613,0.878181,0.868613,0.866253
3,0.112500,0.492470,0.901460,0.901415,0.901460,0.901328


c:\Users\c24082331\OneDrive - Cardiff University\Desktop\RA(UniversalCEFR)\development\universalcefr\Lib\site-packages\sklearn\metrics\_classification.py:1706: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


TrainOutput(global_step=414, training_loss=0.28724858467129694, metrics={'train_runtime': 2418.1279, 'train_samples_per_second': 1.362, 'train_steps_per_second': 0.171, 'total_flos': 230831713194120.0, 'train_loss': 0.28724858467129694, 'epoch': 3.0})

In [23]:
metrics = trainer.evaluate()
all_results.append(metrics)

In [24]:
df = pd.DataFrame(all_results)
print("\nAverage over 5 folds:")
print(df[["eval_accuracy","eval_f1_weighted"]].mean())


Average over 5 folds:
eval_accuracy       0.901460
eval_f1_weighted    0.901328
dtype: float64
